# Hybrid (dense + BM25) retrieval in local search

Dense embeddings match *meaning*. BM25 matches *the exact string*. Running both and
fusing the rankings covers the cases where meaning alone is not enough.

Two things must be wired up, and missing either one **silently** degrades to
dense-only — no error, no warning, just worse retrieval:

1. **A sparse-capable vector store.** `QdrantVectorDBStorage` with
   `sparse_type="bm25"` is the only option. The default `NanoVectorDBStorage` logs
   `"NanoVDB does not support sparse embeddings. Ignoring"` and drops them — the
   build still succeeds.
2. **The sparse embedder in both places.** On `KnowledgeGraph` it produces the
   document-side term weights at index time; on the search engine it produces the
   query-side ones. Pass it to only one and one side of the comparison has no
   sparse vector.

Fusion itself happens server-side in Qdrant, via reciprocal-rank fusion.

**Environment:** `OPENAI_API_KEY`, `LLM_MODEL_NAME`, `EMBEDDER_MODEL_NAME`, and
optionally `OPENAI_BASE_URL` and `QDRANT_URL` (without the latter Qdrant runs
in-memory and nothing persists).

## When hybrid earns its keep

The question to ask is not "is my corpus technical?" but **"do my users type
strings that must match exactly?"** Dense retrieval maps text to a point in
semantic space, and that mapping is lossy in exactly one direction: tokens whose
*identity* carries the meaning, rather than their sense, get blurred into their
neighbours.

Reach for hybrid when the corpus contains:

| case | example | why dense struggles |
|---|---|---|
| **Statute and clause numbers** | «статья 159 УК РФ», «п. 2 ст. 15 ГК» | `159` and `158` sit next to each other in embedding space; the number *is* the query |
| **Part, model, SKU codes** | `A1502`, `MX-2340N`, `ISO 9001` | alphanumeric codes have no learned semantics at all |
| **Numbers that must be literal** | amounts, years, versions, percentages | `v2.1.4` and `v2.4.1` embed almost identically |
| **Rare proper nouns** | surnames, small towns, project code names | rare tokens are under-trained, so their vectors are noisy |
| **Acronyms and abbreviations** | `ОСАГО`, `НДС`, `CRDT` | short strings carry little signal for the encoder |
| **Error and status codes** | `ORA-01555`, `HTTP 429` | pure identifiers |
| **Names that collide with common words** | a product called «Заря», `Rust`, `Go` | the encoder pulls them toward the everyday sense |

The common shape: **a token whose value must be matched character by character,
where a near-miss is not a partial answer but a wrong one.** Citing article 158
when the user asked about 159 is worse than citing nothing.

### When it is not worth it

- **Conceptual, paraphrase-heavy questions** — "why did the project fail?",
  "summarize the arguments against". There is no literal anchor to match, and BM25
  contributes noise that RRF then has to be diluted by.
- **Cross-lingual retrieval** — BM25 matches surface forms, so it does nothing when
  the query and the document are in different languages.
- **Heavily normalized short text** — if entity names were canonicalized at index
  time, the dense side already finds them.

### Costs, so the decision is informed

- Qdrant becomes mandatory; the zero-dependency default store is out.
- Index time grows: every entity, relation and chunk gets a second vector.
- Storage grows by the sparse vectors.
- The BM25 stemmer is language-specific — set `Settings.language` before
  constructing it, or you will stem Russian text with English rules.

In [ ]:
import os
from pathlib import Path

from ragu import (
    ArtifactsExtractorLLM,
    BuilderArguments,
    KnowledgeGraph,
    LocalSearchEngine,
    Settings,
    SimpleChunker,
    StorageArguments,
)
from ragu.models.embedder import EmbedderOpenAI
from ragu.models.llm import LLMOpenAI
from ragu.models.openai import CachedAsyncOpenAI
from ragu.models.sparse_embedder import BM25
from ragu.search_engine.local_search import LocalParams
from ragu.storage.vdb_storage_adapters.qdrant_vdb import QdrantVectorDBStorage
from ragu.utils.ragu_utils import read_text_from_files

DATA_DIR = Path("data/en")

# Phrased to reward lexical matching: it names the entities literally rather than
# describing them.
QUESTION = "What did Dennis Ritchie create at Bell Labs?"

## Models

BM25 picks up its stemmer and stopword list from `Settings.language`, so set the
language before constructing it.

In [ ]:
Settings.language = "english"
Settings.storage_folder = "ragu_working_dir/hybrid_local_search_example"

client = CachedAsyncOpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    api_key=os.environ["OPENAI_API_KEY"],
    rate_max_simultaneous=10,
    rate_max_per_minute=100,
)
llm = LLMOpenAI(client=client, model_name=os.environ["LLM_MODEL_NAME"])
embedder = EmbedderOpenAI(client=client, model_name=os.environ["EMBEDDER_MODEL_NAME"])
await embedder.initialize()

sparse_embedder = BM25()

## Index side

`QdrantVectorDBStorage.location` accepts either `":memory:"` or a URL, so one
expression covers both deployments. With `sparse_embedder` set on the graph, every
entity, relation and chunk gets a BM25 vector alongside its dense one.

This is the expensive cell — run it once.

In [ ]:
storage_settings = StorageArguments(
    vdb_storage_type=QdrantVectorDBStorage,
    vdb_storage_kwargs={
        "sparse_type": "bm25",
        "location": os.environ.get("QDRANT_URL", ":memory:"),
    },
)

knowledge_graph = KnowledgeGraph(
    llm=llm,
    embedder=embedder,
    sparse_embedder=sparse_embedder,
    chunker=SimpleChunker(max_chunk_size=1000),
    artifact_extractor=ArtifactsExtractorLLM(llm=llm, embedder=embedder),
    builder_settings=BuilderArguments(),
    storage_settings=storage_settings,
)
await knowledge_graph.build_from_docs(read_text_from_files(DATA_DIR))
await knowledge_graph.reindex_graph()

## Query side

The same `sparse_embedder` instance goes to the engine. This is the half that is
easy to forget: the graph would still be indexed with BM25 vectors, the queries
just would not carry one, and Qdrant would quietly run a dense-only search.

In [ ]:
engine = LocalSearchEngine(
    llm=llm,
    knowledge_graph=knowledge_graph,
    embedder=embedder,
    sparse_embedder=sparse_embedder,
)

response = await engine.query(QUESTION, LocalParams(top_k=10))
print(response.response)

print("\nentities retrieved:")
for entity in response.retrieval.result.entities:
    print(f"  - {entity.entity_name} ({entity.entity_type})")

## Choosing the sparse model

`sparse_type` on the store and the class you construct must agree — the store
declares the vector field, the embedder fills it.

| model | `sparse_type` | notes |
|---|---|---|
| `BM25` | `"bm25"` | Lexical, language-aware stemming, no neural weights. The default choice, and the right one for identifiers and numbers. |
| `BM42` | `"bm42"` | Learned term weights from transformer attention. Better on short text; needs its own model weights, and has no stemmer or stopword knobs. |
| `SPLADE` | `"splade"` | Learned sparse expansion, adds related terms. Strongest on English; the default checkpoint is English-only. |

`BM25` also takes the classic knobs — `k` (term-frequency saturation), `b` (length
normalization) and `avg_len`.

In [ ]:
russian_bm25 = BM25(language="russian", k=1.2, b=0.75, avg_len=256.0)
print(f"configured: {type(russian_bm25).__name__} for russian")

## Tuning the built-in stemmer

`BM25` forwards a handful of arguments to FastEmbed. They fall into two groups:
ones that change **which terms exist**, and ones that change **how they are
weighted**.

| argument | default | effect |
|---|---|---|
| `language` | `Settings.language` | Picks both the Snowball stemmer and the stopword list |
| `disable_stemmer` | `False` | Turns stemming off; stopword and length filtering still run |
| `token_max_length` | `40` | Tokens longer than this are silently dropped |
| `k` | `1.2` | Term-frequency saturation — how fast repeats stop helping |
| `b` | `0.75` | Length normalization — how much long documents are penalized |
| `avg_len` | `256.0` | Assumed average document length, used together with `b` |
| `model_name` | `"Qdrant/bm25"` | Which stopword/stemmer bundle to download |
| `cache_dir` | `None` | Where those files are cached |

### `language` selects the stemmer and the stopword list

In [ ]:
russian_bm25 = BM25(language="russian")
english_bm25 = BM25(language="english")


def term_ids(sparse_embedder: BM25, text: str) -> set[int]:
    """Term ids FastEmbed produces for one text."""
    return set(sparse_embedder.embed_document([text])[0].indices)


# Five words, one of which ("и") is a Russian stopword.
sentence = "Регулируется статьями и нормами закона"
print(f"{'language':10} {'terms':>6}  {'форма merge':>12}")
for name, bm25 in (("russian", russian_bm25), ("english", english_bm25)):
    shared = len(term_ids(bm25, "статья") & term_ids(bm25, "статьями"))
    print(f"{name:10} {len(term_ids(bm25, sentence)):>6}  {shared:>12}")

`russian` drops `и` as a stopword, leaving four terms where `english` keeps all
five. And `статья` / `статьями` share a term only under `russian` — the English
Snowball rules leave Cyrillic untouched, so every word form stays its own term.

Setting the wrong language is therefore not a small mistake: Russian text scored
with the English list keeps every Russian stopword as a real term and gets no
stemming at all. `BM25()` inherits `Settings.language` when you do not pass
`language`, so set the global before constructing it.

### `token_max_length` drops long tokens silently

In [ ]:
for limit in (40, 64):
    bm25 = BM25(language="russian", token_max_length=limit)
    long_token = "A" * 45
    kept = len(bm25.embed_document([long_token])[0].indices)
    print(f"token_max_length={limit:3} -> 45-character token yields {kept} term(s)")

The default of 40 is generous for words and tight for machine-generated strings —
long SKUs, hashes, concatenated codes. There is no warning when a token is dropped,
so raise it if your identifiers are long.

### `k`, `b` and `avg_len` change weights, not terms

These are the classic BM25 scoring parameters. They do not add or remove terms, so
they cannot fix a retrieval miss — they only reorder hits that were already found.

In [ ]:
document = ["Мошенничество регулируется статьёй 159 УК РФ и связанными нормами."]
default_weights = BM25(language="russian").embed_document(document)[0]
tuned_weights = BM25(language="russian", k=2.0, b=0.3, avg_len=64.0).embed_document(document)[0]

print(f"same terms:  {set(default_weights.indices) == set(tuned_weights.indices)}")
print(f"default k/b: {[round(v, 4) for v in default_weights.values[:4]]}")
print(f"tuned   k/b: {[round(v, 4) for v in tuned_weights.values[:4]]}")

Lower `b` penalizes long documents less, which helps when your chunks vary a lot in
size. Higher `k` lets repeated terms keep accumulating weight. Set `avg_len` near
your actual mean chunk length in tokens — leaving it at 256 while chunking at 1000
characters makes `b` behave as if every document were long.

## Verifying it is actually hybrid

The failure mode is silent, so assert the wiring instead of trusting it. Three
things have to be true at once, and each is a public attribute:

In [ ]:
store = knowledge_graph.index.chunks_vector_db

checks = {
    "store supports sparse": isinstance(store, QdrantVectorDBStorage),
    "index side wired": knowledge_graph.sparse_embedder is not None,
    "query side wired": engine.retriever.sparse_embedder is not None,
    "same model both sides": knowledge_graph.sparse_embedder is engine.retriever.sparse_embedder,
}
for label, ok in checks.items():
    print(f"  [{'x' if ok else ' '}] {label}")

print(f"\nhybrid active: {all(checks.values())}")

In [ ]:
await knowledge_graph.index.close()